# Continuous Path Architecture (CPA) - Validation

Training an end-to-end CPA model on a character-level Shakespeare dataset to validate the Velocity Flow Matching objective.


## 1. Environment & Imports


In [1]:
!uv pip install -q jax jaxlib equinox optax requests


In [2]:
import os
import requests
import jax
import jax.numpy as jnp
from jax import lax
import equinox as eqx
import optax
import time
import functools


## 2. Dataset & Tokenizer

We download `tiny-shakespeare.txt` and create a character-level tokenizer. We also define a dataloader that creates prefix paths for Strategy A.


In [11]:
import jax.numpy as jnp # Kept for the jnp.array casting below

# 1. Load dataset directly from Kaggle's local file system
text_file = "/kaggle/input/datasets/aniruddhavarma/shakespeare-text/shakespeare_copy.txt"

with open(text_file, "r", encoding="utf-8") as f:
    text = f.read()

# 2. Character-level tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
char2int = {c: i for i, c in enumerate(chars)}
int2char = {i: c for i, c in enumerate(chars)}

encode = lambda s: [char2int[c] for c in s]
decode = lambda l: ''.join([int2char[i] for i in l])

print(f"Dataset length: {len(text)} chars")
print(f"Vocabulary size: {vocab_size}")

# 3. Encode and cast to JAX array
data = jnp.array(encode(text), dtype=jnp.int32)

# 4. Training split (90% train, 10% validation)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

Dataset length: 5436475 chars
Vocabulary size: 84


In [12]:
# Strategy A Dataloader: Deterministic sequential batching
def get_batches(split, batch_size, min_len=4, max_len=64, key=None):
    dataset = train_data if split == 'train' else val_data
    import numpy as np
    dataset_np = np.array(dataset)
    
    # We step by max_len for non-overlapping sequential blocks
    step = max_len
    indices = np.arange(0, len(dataset_np) - max_len - 1, step)
    num_batches = len(indices) // batch_size
    
    for i in range(num_batches):
        x_np = np.zeros((batch_size, max_len), dtype=np.int32)
        y_np = np.zeros((batch_size,), dtype=np.int32)
        mask_np = np.zeros((batch_size, max_len), dtype=np.bool_)
        
        if key is not None:
            k1, key = jax.random.split(key)
            prefix_lens_np = np.array(jax.random.randint(k1, (batch_size,), min_len, max_len))
        else:
            prefix_lens_np = np.full((batch_size,), max_len, dtype=np.int32)
            
        batch_indices = indices[i * batch_size : (i + 1) * batch_size]
        
        for b in range(batch_size):
            idx = batch_indices[b]
            plen = prefix_lens_np[b]
            
            x_np[b, :plen] = dataset_np[idx : idx + plen]
            mask_np[b, :plen] = True
            y_np[b] = dataset_np[idx + plen]
            
        yield jnp.array(x_np), jnp.array(y_np), jnp.array(mask_np)


## 3. CPA Primitives

These are the mathematically rigorous Spline and CDF operations for Continuous Paths.


In [13]:
def compute_anchors(lexical_increments: jnp.ndarray, mask: jnp.ndarray = None) -> jnp.ndarray:
    if mask is not None:
        lexical_increments = lexical_increments * mask[..., None]
    origin = jnp.zeros_like(lexical_increments[:, :1, :])
    return jnp.concatenate([origin, jnp.cumsum(lexical_increments, axis=1)], axis=1)

def _catmull_rom_tangents(anchors: jnp.ndarray) -> jnp.ndarray:
    tangents = jnp.zeros_like(anchors)
    tangents = tangents.at[:, 1:-1].set((anchors[:, 2:] - anchors[:, :-2]) / 2.0)
    tangents = tangents.at[:, 0].set(anchors[:, 1] - anchors[:, 0])
    tangents = tangents.at[:, -1].set(anchors[:, -1] - anchors[:, -2])
    return tangents

def _hermite_basis(t: jnp.ndarray):
    t2 = t * t
    t3 = t2 * t
    return (2 * t3 - 3 * t2 + 1, t3 - 2 * t2 + t, -2 * t3 + 3 * t2, t3 - t2)

def evaluate_hermite(anchors, tangents, t: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    _B, K, _d = anchors.shape
    # Instead of scaling by the static max_len, scale by the actual sequence length
    plen = jnp.sum(mask, axis=-1) # shape: (B,)
    t_scaled = t[None, :] * plen[:, None] # shape: (B, M)
    
    seg = jnp.floor(t_scaled).astype(jnp.int32).clip(0, anchors.shape[1] - 2)
    u = t_scaled - seg.astype(t.dtype)

    p0 = anchors[jnp.arange(_B)[:, None], seg]
    p1 = anchors[jnp.arange(_B)[:, None], seg + 1]
    m0 = tangents[jnp.arange(_B)[:, None], seg]
    m1 = tangents[jnp.arange(_B)[:, None], seg + 1]

    h00, h10, h01, h11 = _hermite_basis(u)
    h00, h10, h01, h11 = (x[:, :, None] for x in (h00, h10, h01, h11))
    return h00 * p0 + h10 * m0 + h01 * p1 + h11 * m1

def _catmull_rom_tangents_masked(anchors: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    tangents = jnp.zeros_like(anchors)
    tangents = tangents.at[:, 1:-1].set((anchors[:, 2:] - anchors[:, :-2]) / 2.0)
    tangents = tangents.at[:, 0].set(anchors[:, 1] - anchors[:, 0])
    
    # Fix boundary: tangent at the last valid anchor should be the forward difference of the last step
    plen = jnp.sum(mask, axis=-1)
    _B = anchors.shape[0]
    b_idx = jnp.arange(_B)
    
    # default end tangent
    tangents = tangents.at[:, -1].set(anchors[:, -1] - anchors[:, -2])
    
    # overwrite the tangent at plen with the forward difference to prevent curving back to zero
    last_tangent = anchors[b_idx, plen] - anchors[b_idx, plen - 1]
    tangents = tangents.at[b_idx, plen].set(last_tangent)
    
    return tangents

def initialize_path_hermite(lexical_increments, M, mask=None):
    anchors = compute_anchors(lexical_increments, mask)
    if mask is None:
        mask = jnp.ones(lexical_increments.shape[:-1], dtype=jnp.bool_)
        
    tangents = _catmull_rom_tangents_masked(anchors, mask)
    t = jnp.linspace(0.0, 1.0, M)
    return evaluate_hermite(anchors, tangents, t, mask)


In [14]:
def speed_density(gamma: jnp.ndarray, w: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    logits = jnp.einsum("bmd,d->bm", gamma, w) + b
    return jax.nn.softplus(logits) + 0.05

def build_cdf(rho: jnp.ndarray):
    B, M = rho.shape
    dt = 1.0 / (M - 1)
    pieces = (rho[:, :-1] + rho[:, 1:]) / 2.0 * dt
    cdf_tail = jnp.cumsum(pieces, axis=1)
    cdf = jnp.concatenate([jnp.zeros((B, 1), dtype=rho.dtype), cdf_tail], axis=1)
    
    total = cdf[:, -1:]
    cdf = cdf / jnp.maximum(total, 1e-8)
    cdf = cdf.at[:, 0].set(0.0)
    cdf = cdf.at[:, -1].set(1.0)
    return cdf, total

def _resample_single(cdf: jnp.ndarray, path: jnp.ndarray, target: jnp.ndarray) -> jnp.ndarray:
    t_orig = jnp.linspace(0.0, 1.0, cdf.shape[0])
    new_t = jnp.interp(target, cdf, t_orig)
    resampled = jax.vmap(lambda col: jnp.interp(new_t, t_orig, col), in_axes=1, out_axes=1)(path)
    return resampled

def reparametrize_path(gamma: jnp.ndarray, w: jnp.ndarray, b: jnp.ndarray):
    # 1. Compute density on coordinates
    rho = speed_density(gamma, w, b)
    cdf, total_density = build_cdf(rho)
    target = jnp.linspace(0.0, 1.0, gamma.shape[1])
    
    # 2. Resample the path geometry
    gamma_out = jax.vmap(functools.partial(_resample_single, target=target))(cdf, gamma)
    boundary_pullback = rho[:, -1:] / jnp.maximum(total_density, 1e-8)
    return gamma_out, boundary_pullback


## 4. CPA Network (Equinox)

We build the Deep Path Processing layer: Path-Integral Attention + MLP + Time-Warp.


In [15]:
class PathIntegralAttention(eqx.Module):
    w_q: jnp.ndarray
    w_k: jnp.ndarray
    w_v: jnp.ndarray
    alpha: float
    log_beta: jnp.ndarray

    def __init__(self, d, key):
        k1, k2, k3 = jax.random.split(key, 3)
        self.w_q = jax.random.normal(k1, (d, d)) / jnp.sqrt(d)
        self.w_k = jax.random.normal(k2, (d, d)) / jnp.sqrt(d)
        self.w_v = jax.random.normal(k3, (d, d)) * 0.02
        
        # RESTORED: Standard scaled dot-product temperature
        self.alpha = 1.0 / jnp.sqrt(d) 
        self.log_beta = jnp.array(0.54) 

    def __call__(self, gamma):
        M = gamma.shape[1]
        dt = 1.0 / (M - 1)
        
        # True derivatives from path
        sequence_tangent = jnp.zeros_like(gamma)
        sequence_tangent = sequence_tangent.at[:, 1:-1].set((gamma[:, 2:] - gamma[:, :-2]) / (2.0 * dt))
        sequence_tangent = sequence_tangent.at[:, 0].set((gamma[:, 1] - gamma[:, 0]) / dt)
        sequence_tangent = sequence_tangent.at[:, -1].set((gamma[:, -1] - gamma[:, -2]) / dt)

        # Global Path Normalization
        sigma_global = jnp.sqrt(jnp.mean(jnp.sum(sequence_tangent**2, axis=-1, keepdims=True), axis=1, keepdims=True) + 1e-8)
        normalized_sequence_tangent = sequence_tangent / sigma_global
        
        # Values are tangent vectors, never absolute path coordinates
        V = normalized_sequence_tangent @ self.w_v
        
        Q = normalized_sequence_tangent @ self.w_q
        K = normalized_sequence_tangent @ self.w_k
        vel_align = jnp.einsum("btd,bsd->bts", Q, K) * self.alpha
        
        # Distance computed on coordinates and divided by d. 
        dist = jnp.sum((gamma[:, :, None, :] - gamma[:, None, :, :])**2, axis=-1) / gamma.shape[-1]
        
        logits = vel_align - jax.nn.softplus(self.log_beta) * dist
        weights = jax.nn.softmax(logits, axis=-1)
        
        # Return the update for ODE integration
        return jnp.einsum("bts,bsd->btd", weights, V)
class ContinuousPathLayer(eqx.Module):
    attention: PathIntegralAttention
    mlp_in: eqx.nn.Linear
    mlp_out: eqx.nn.Linear
    reparam_w: jnp.ndarray
    reparam_b: jnp.ndarray
    dt: float

    def __init__(self, d, key, dt):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.attention = PathIntegralAttention(d, k1)
        self.mlp_in = eqx.nn.Linear(d, 4 * d, key=k2)
        self.mlp_out = eqx.nn.Linear(4 * d, d, key=k3)
        self.reparam_w = jax.random.normal(k4, (d,)) * 0.02
        self.reparam_b = jnp.array(0.1)
        self.dt = dt

    def __call__(self, gamma):
        # 1. Attention
        attention_depth_update = self.attention(gamma)
        gamma = gamma + self.dt * attention_depth_update
        
        # 2. MLP
        mlp_h = jax.vmap(jax.vmap(self.mlp_in))(gamma)
        mlp_h = jax.nn.gelu(mlp_h)
        mlp_depth_update = jax.vmap(jax.vmap(self.mlp_out))(mlp_h)
        gamma = gamma + self.dt * mlp_depth_update
        
        # 3. Reparametrization
        gamma_out, boundary_pullback = reparametrize_path(gamma, self.reparam_w, self.reparam_b)
        
        return gamma_out, boundary_pullback

class CPAModel(eqx.Module):
    lexical_increments: jnp.ndarray
    layers: list
    inverse_map_mlp_in: eqx.nn.Linear
    inverse_map_mlp_out: eqx.nn.Linear
    d: int
    M: int
    dt: float

    def __init__(self, vocab_size, d, M, L, key):
        keys = jax.random.split(key, 3 + L)
        self.d = d
        self.M = M
        
        # Each token is a lexical increment along the initial path
        self.lexical_increments = jax.random.normal(keys[0], (vocab_size, d)) * 0.1
        
        
        self.dt = 1.0 / L
        self.layers = [ContinuousPathLayer(d, keys[i+1], self.dt) for i in range(L)]
        
        # Learn an approximate inverse map for depth transformations at the boundary
        self.inverse_map_mlp_in = eqx.nn.Linear(d, d, key=keys[-2])
        self.inverse_map_mlp_out = eqx.nn.Linear(d, d * d, key=keys[-1])

    def get_lexical_increments(self):
        return self.lexical_increments

    def __call__(self, tokens, mask):
        # Tokens: (B, N) int array. 
        # Extract lexical increments
        lexical_increment_seq = self.get_lexical_increments()[tokens] # (B, N, d)
        
        # Phase 1: Spline Init
        gamma = initialize_path_hermite(lexical_increment_seq, self.M, mask)
        
        # Phase 2: Deep Path Processing
        boundary_pullback = jnp.ones((tokens.shape[0], 1), dtype=gamma.dtype)
        for layer in self.layers:
            gamma, layer_boundary_pullback = layer(gamma)
            boundary_pullback = boundary_pullback * layer_boundary_pullback
            
        # Phase 3: Pull the terminal sequence tangent back to lexical space
        gamma_end = gamma[:, -1, :]
        dt = 1.0 / (self.M - 1)
        output_sequence_tangent = (gamma[:, -1, :] - gamma[:, -2, :]) / dt
        
        # Undo each normalized CDF warp, then convert normalized-time tangent to a per-token increment
        seq_len = jnp.sum(mask, axis=-1, keepdims=True)
        lexical_scale_tangent = output_sequence_tangent * boundary_pullback / seq_len
        
        h = jax.nn.gelu(jax.vmap(self.inverse_map_mlp_in)(gamma_end))
        approximate_inverse_map = jax.vmap(self.inverse_map_mlp_out)(h).reshape(-1, self.d, self.d)
        delta_P = jnp.einsum("bij,bj->bi", approximate_inverse_map, lexical_scale_tangent)
        
        return delta_P

def lexical_distance_logits(delta_P, lexical_increments):
    return -jnp.sum((delta_P[:, None, :] - lexical_increments[None, :, :])**2, axis=-1)

def run_invariant_tests():
    test_M = 257
    dt = 1.0 / (test_M - 1)
    t = jnp.linspace(0.0, 1.0, test_M)

    # Identity path: a uniform reparameterization must preserve the samples exactly.
    identity_path = t[None, :, None]
    identity_out, _ = reparametrize_path(identity_path, jnp.zeros((1,)), jnp.array(0.0))
    assert bool(jnp.allclose(identity_out, identity_path, atol=1e-6))

    # Constant density: geometry is preserved and rho_end / Z is one.
    curved_path = jnp.stack([t, t**2], axis=-1)[None, :, :]
    uniform_out, uniform_pullback = reparametrize_path(curved_path, jnp.zeros((2,)), jnp.array(0.3))
    assert bool(jnp.allclose(uniform_out, curved_path, atol=1e-6))
    assert bool(jnp.allclose(uniform_pullback, jnp.ones_like(uniform_pullback), atol=1e-6))

    # Known nonconstant density: undo the inverse-CDF boundary pacing independently.
    nonuniform_out, nonuniform_pullback = reparametrize_path(identity_path, jnp.array([1.7]), jnp.array(-0.4))
    output_boundary_tangent = (nonuniform_out[:, -1, :] - nonuniform_out[:, -2, :]) / dt
    recovered_boundary_tangent = output_boundary_tangent * nonuniform_pullback
    assert bool(jnp.allclose(recovered_boundary_tangent, jnp.ones_like(recovered_boundary_tangent), atol=2e-2, rtol=2e-2))

    # Sequence-length independence: normalized-time tangent divided by N recovers one lexical increment.
    increment = jnp.array([[[0.3, -0.2]]])
    short_seq = jnp.tile(increment, (1, 4, 1))
    long_seq = jnp.tile(increment, (1, 8, 1))
    short_path = initialize_path_hermite(short_seq, test_M, jnp.ones((1, 4), dtype=jnp.bool_))
    long_path = initialize_path_hermite(long_seq, test_M, jnp.ones((1, 8), dtype=jnp.bool_))
    short_increment = (short_path[:, -1, :] - short_path[:, -2, :]) / dt / 4.0
    long_increment = (long_path[:, -1, :] - long_path[:, -2, :]) / dt / 8.0
    assert bool(jnp.allclose(short_increment, long_increment, atol=1e-5))
    assert bool(jnp.allclose(short_increment, increment[:, 0, :], atol=1e-5))

    # Decoding agreement: CE argmax and generation nearest-neighbor lookup select the same token.
    predictions = jnp.array([[0.0, 0.0], [1.8, 0.1]])
    vocabulary = jnp.array([[0.0, 0.1], [1.0, 1.0], [2.0, 0.0]])
    logits = lexical_distance_logits(predictions, vocabulary)
    nearest = jnp.argmin(jnp.sum((vocabulary[None, :, :] - predictions[:, None, :])**2, axis=-1), axis=-1)
    assert bool(jnp.array_equal(jnp.argmax(logits, axis=-1), nearest))

    return "identity, constant density, nonconstant density, sequence length, and decode agreement passed"

print("Invariant tests:", run_invariant_tests())

## 5. Flow Matching Loss & Training Loop


Invariant tests: identity, constant density, nonconstant density, sequence length, and decode agreement passed


In [16]:
# Hyperparameters
d = 64
M = 32
L = 4
batch_size = 64
learning_rate = 1e-3
epochs = 50000

key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

model = CPAModel(vocab_size=vocab_size, d=d, M=M, L=L, key=k1)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate)
)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
@eqx.filter_jit
def loss_fn(model, x, y, mask):
    delta_P = model(x, mask) # (B, d) lexical displacement
    logits = lexical_distance_logits(delta_P, model.get_lexical_increments())
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    return jnp.mean(loss)
def compute_diagnostics(model, grads, x, y, mask):
    # 1. Embedding Health (Fatal)
    V_emb = model.get_lexical_increments()
    norm_V = V_emb / jnp.maximum(jnp.linalg.norm(V_emb, axis=-1, keepdims=True), 1e-8)
    cosine_sim = norm_V @ norm_V.T
    mean_cosine = (jnp.sum(cosine_sim) - V_emb.shape[0]) / (V_emb.shape[0] * (V_emb.shape[0] - 1))
    
    S = jnp.linalg.svd(V_emb, compute_uv=False)
    S_norm = S / jnp.sum(S)
    effective_rank = jnp.exp(-jnp.sum(S_norm * jnp.log(jnp.maximum(S_norm, 1e-8))))
    
    dists_emb = jnp.sum((V_emb[:, None, :] - V_emb[None, :, :])**2, axis=-1)
    dists_emb = dists_emb + jnp.eye(V_emb.shape[0]) * 1e6
    mean_nn_dist = jnp.mean(jnp.min(dists_emb, axis=-1))
    
    # 2. Gradient Health (Fatal)
    reparam_grad_norm = jnp.linalg.norm(grads.layers[0].reparam_w)
    emb_grad_norm = jnp.linalg.norm(grads.lexical_increments)
    
    total_grad_sq = sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads) if g is not None)
    net_grad_norm = jnp.sqrt(jnp.maximum(total_grad_sq - emb_grad_norm**2, 1e-8))
    emb_grad_ratio = emb_grad_norm / jnp.maximum(net_grad_norm, 1e-8)
    
    # 3. Path & Attention Diagnostics (Severe)
    lexical_increment_seq = model.get_lexical_increments()[x]
    gamma = initialize_path_hermite(lexical_increment_seq, model.M, mask)
    path_energy_in = jnp.mean(jnp.sum(gamma**2, axis=-1))
    
    attn = model.layers[0].attention
    M = gamma.shape[1]
    dt = 1.0 / (M - 1)
    sequence_tangent = jnp.zeros_like(gamma)
    sequence_tangent = sequence_tangent.at[:, 1:-1].set((gamma[:, 2:] - gamma[:, :-2]) / (2.0 * dt))
    sequence_tangent = sequence_tangent.at[:, 0].set((gamma[:, 1] - gamma[:, 0]) / dt)
    sequence_tangent = sequence_tangent.at[:, -1].set((gamma[:, -1] - gamma[:, -2]) / dt)
    
    sigma_global = jnp.sqrt(jnp.mean(jnp.sum(sequence_tangent**2, axis=-1, keepdims=True), axis=1, keepdims=True) + 1e-8)
    normalized_sequence_tangent = sequence_tangent / sigma_global
    
    Q = normalized_sequence_tangent @ attn.w_q
    K = normalized_sequence_tangent @ attn.w_k
    vel_align = jnp.einsum('btd,bsd->bts', Q, K) * attn.alpha
    
    # UPDATED: Compute diagnostic dist precisely the way the layer does
    d = gamma.shape[-1]
    dist = jnp.sum((gamma[:, :, None, :] - gamma[:, None, :, :])**2, axis=-1) / d
    
    logits = vel_align - jax.nn.softplus(attn.log_beta) * dist
    weights = jax.nn.softmax(logits, axis=-1)
    
    entropy = -jnp.sum(weights * jnp.log(jnp.maximum(weights, 1e-8)), axis=-1)
    mean_attn_entropy = jnp.mean(entropy)
    
    # DomRatio is now correctly scaled and meaningful
    dom_ratio = jnp.mean(jnp.abs(vel_align)) / jnp.maximum(jnp.mean(dist), 1e-8)
    
    for l in model.layers:
        gamma, _ = l(gamma)
    path_energy_out = jnp.mean(jnp.sum(gamma**2, axis=-1))
    
    return {
        'eff_rank': effective_rank,
        'mean_cos': mean_cosine,
        'mean_nn_dist': mean_nn_dist,
        'reparam_grad': reparam_grad_norm,
        'emb_grad_ratio': emb_grad_ratio,
        'attn_entropy_l0': mean_attn_entropy,
        'attn_dom_ratio_l0': dom_ratio,
        'path_energy_in': path_energy_in,
        'path_energy_out': path_energy_out
    }

@eqx.filter_jit
def step(model, opt_state, x, y, mask):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y, mask)
    metrics = compute_diagnostics(model, grads, x, y, mask)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss, metrics



## 6. Autoregressive Generation (Pre-Training)

Let's see what the model generates before any training.


In [17]:
def generate(model, prompt, max_new_tokens=50):
    tokens = encode(prompt)
    
    for _ in range(max_new_tokens):
        # We process the prefix
        x = jnp.array([tokens], dtype=jnp.int32)
        mask = jnp.ones_like(x, dtype=jnp.bool_)
        
        # Predict the next lexical displacement
        delta_P = model(x, mask)[0] # (d,)
        
        # Geometric Vocabulary Matching (L2 distance)
        dists = jnp.sum((model.get_lexical_increments() - delta_P)**2, axis=-1)
        next_token = jnp.argmin(dists).item()
        
        tokens.append(next_token)
        
    return decode(tokens)

print("Generation Before Training (Random):")
out = generate(model, "ROMEO ")
print(out)


Generation Before Training (Random):
ROMEO `)t>1)t>}?i21NM?!O1NM?!Jq_j,u&1NM?!_j,1<<TT}}7rN1?


## 7. Training Loop


In [20]:
from tqdm import tqdm

print("Starting training...")
start_time = time.time()
key = k2

# One epoch over the dataset
epochs_to_run = 50

for epoch in range(epochs_to_run):
    key, subkey = jax.random.split(key)
    
    # Calculate total batches for tqdm
    dataset = train_data
    num_batches = (len(dataset) - 64 - 1) // 64 // batch_size
    batch_iterator = get_batches('train', batch_size, max_len=64, key=subkey)
    
    total_epoch_loss = 0.0
    num_steps = 0
    
    pbar = tqdm(batch_iterator, total=num_batches, desc=f"Epoch {epoch+1}/{epochs_to_run}")
    for step_idx, (x, y, mask) in enumerate(pbar):
        model, opt_state, loss, metrics = step(model, opt_state, x, y, mask)
        
        loss_val = float(loss)
        total_epoch_loss += loss_val
        num_steps += 1
        
        pbar.set_postfix(loss=f"{loss_val:.4f}", avg=f"{total_epoch_loss / num_steps:.4f}")
        
        if step_idx > 0 and (step_idx % 200 == 0 or step_idx == num_batches - 1):
            print(f"\nStep {step_idx:4d} | Batch Loss: {loss_val:.4f} | Avg Loss: {total_epoch_loss / num_steps:.4f} | Time: {time.time() - start_time:.1f}s")
            print(f"  [Fatal]  EffRank: {metrics['eff_rank']:.1f} | MeanCos: {metrics['mean_cos']:.3f} | NNDist: {metrics['mean_nn_dist']:.3f}")
            print(f"  [Fatal]  ReparamGrad: {metrics['reparam_grad']:.4f} | EmbGradRatio: {metrics['emb_grad_ratio']:.2f}")
            print(f"  [Severe] AttnEntropy(L0): {metrics['attn_entropy_l0']:.2f} | DomRatio(L0): {metrics['attn_dom_ratio_l0']:.2f}")
            print(f"  [Severe] PathEnergy: {metrics['path_energy_in']:.2f} -> {metrics['path_energy_out']:.2f}\n")
            
    # Print average epoch loss at the end of the epoch
    avg_epoch_loss = total_epoch_loss / max(num_steps, 1)
    print(f"\n{'='*40}")
    print(f"End of Epoch {epoch+1}")
    print(f"Average Epoch Loss: {avg_epoch_loss:.4f}")
    print(f"Total time elapsed: {time.time() - start_time:.1f}s")
    print(f"{'='*40}\n")


Epoch 49/50:  17%|█▋        | 208/1194 [00:03<00:14, 67.14it/s, avg=1.6570, loss=1.8160]


Step  200 | Batch Loss: 1.5583 | Avg Loss: 1.6551 | Time: 893.0s
  [Fatal]  EffRank: 51.3 | MeanCos: 0.112 | NNDist: 2.342
  [Fatal]  ReparamGrad: 0.6537 | EmbGradRatio: 1.79
  [Severe] AttnEntropy(L0): 3.04 | DomRatio(L0): 1.72
  [Severe] PathEnergy: 51.94 -> 31.48



Epoch 49/50:  34%|███▍      | 411/1194 [00:06<00:11, 67.20it/s, avg=1.6535, loss=1.3325]


Step  400 | Batch Loss: 1.6569 | Avg Loss: 1.6539 | Time: 896.0s
  [Fatal]  EffRank: 51.3 | MeanCos: 0.112 | NNDist: 2.332
  [Fatal]  ReparamGrad: 0.6639 | EmbGradRatio: 1.44
  [Severe] AttnEntropy(L0): 3.08 | DomRatio(L0): 2.23
  [Severe] PathEnergy: 38.48 -> 26.59



Epoch 49/50:  51%|█████▏    | 612/1194 [00:09<00:08, 66.11it/s, avg=1.6470, loss=1.5584]

...


Step  800 | Batch Loss: 1.9142 | Avg Loss: 1.6336 | Time: 919.8s
  [Fatal]  EffRank: 51.2 | MeanCos: 0.112 | NNDist: 2.318
  [Fatal]  ReparamGrad: 0.3411 | EmbGradRatio: 1.88
  [Severe] AttnEntropy(L0): 3.07 | DomRatio(L0): 1.78
  [Severe] PathEnergy: 45.45 -> 26.95



Epoch 50/50:  85%|████████▍ | 1012/1194 [00:15<00:02, 64.12it/s, avg=1.6309, loss=1.4303]


Step 1000 | Batch Loss: 1.7263 | Avg Loss: 1.6321 | Time: 923.0s
  [Fatal]  EffRank: 51.2 | MeanCos: 0.112 | NNDist: 2.320
  [Fatal]  ReparamGrad: 0.3810 | EmbGradRatio: 1.91
  [Severe] AttnEntropy(L0): 3.03 | DomRatio(L0): 1.54
  [Severe] PathEnergy: 55.88 -> 30.71



Epoch 50/50: 100%|██████████| 1194/1194 [00:18<00:00, 65.79it/s, avg=1.6290, loss=1.8741]


Step 1193 | Batch Loss: 1.8741 | Avg Loss: 1.6290 | Time: 926.2s
  [Fatal]  EffRank: 51.2 | MeanCos: 0.112 | NNDist: 2.328
  [Fatal]  ReparamGrad: 0.3196 | EmbGradRatio: 1.95
  [Severe] AttnEntropy(L0): 3.07 | DomRatio(L0): 1.58
  [Severe] PathEnergy: 44.88 -> 28.12


End of Epoch 50
Average Epoch Loss: 1.6290
Total time elapsed: 926.2s



## 8. Autoregressive Generation (Post-Training)

Now let's see what the model generates after training.


In [22]:
print("Generation After Training:")
out = generate(model, "Romeo ")
print(out)


Generation After Training:
Romeo there the the the the shall the have the herst han


## 6. Baseline Transformer Model
To check if the number of parameters is the limiting factor, let's train a standard Transformer with a similar parameter budget (d=64, L=4).

In [ ]:
class TransformerBlock(eqx.Module):
    attention: eqx.nn.MultiheadAttention
    mlp: eqx.nn.MLP
    ln1: eqx.nn.LayerNorm
    ln2: eqx.nn.LayerNorm

    def __init__(self, d, num_heads, key):
        k1, k2 = jax.random.split(key)
        self.attention = eqx.nn.MultiheadAttention(num_heads, d, key=k1)
        self.mlp = eqx.nn.MLP(d, d, 4 * d, 1, key=k2)
        self.ln1 = eqx.nn.LayerNorm(d)
        self.ln2 = eqx.nn.LayerNorm(d)

    def __call__(self, x, mask):
        # x: (seq_len, d)
        # Causal mask
        seq_len = x.shape[0]
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        
        h = jax.vmap(self.ln1)(x)
        attn_out = self.attention(h, h, h, mask=causal_mask)
        x = x + attn_out
        
        h = jax.vmap(self.ln2)(x)
        mlp_out = jax.vmap(self.mlp)(h)
        x = x + mlp_out
        return x

class TransformerModel(eqx.Module):
    token_emb: eqx.nn.Embedding
    pos_emb: eqx.nn.Embedding
    layers: list
    ln_f: eqx.nn.LayerNorm
    head: eqx.nn.Linear
    d: int
    max_len: int

    def __init__(self, vocab_size, d, L, num_heads, max_len, key):
        keys = jax.random.split(key, 3 + L)
        self.d = d
        self.max_len = max_len
        self.token_emb = eqx.nn.Embedding(vocab_size, d, key=keys[0])
        self.pos_emb = eqx.nn.Embedding(max_len, d, key=keys[1])
        self.layers = [TransformerBlock(d, num_heads, keys[i+2]) for i in range(L)]
        self.ln_f = eqx.nn.LayerNorm(d)
        self.head = eqx.nn.Linear(d, vocab_size, key=keys[-1])

    def __call__(self, tokens):
        # tokens: (seq_len,)
        seq_len = tokens.shape[0]
        pos = jnp.arange(seq_len)
        
        x = jax.vmap(self.token_emb)(tokens) + jax.vmap(self.pos_emb)(pos)
        
        for layer in self.layers:
            x = layer(x, None)
            
        x = jax.vmap(self.ln_f)(x)
        logits = jax.vmap(self.head)(x)
        return logits # (seq_len, vocab_size)


In [26]:
tf_model = TransformerModel(vocab_size=vocab_size, d=d, L=L, num_heads=4, max_len=256, key=k1)

tf_optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate)
)
tf_opt_state = tf_optimizer.init(eqx.filter(tf_model, eqx.is_array))

@eqx.filter_jit
def tf_loss_fn(model, x, y, mask):
    # x: (B, seq_len), y: (B,), mask: (B, seq_len)
    def single_example_loss(x_i, y_i, mask_i):
        seq_len = jnp.sum(mask_i)
        logits = model(x_i) # (max_len, vocab_size)
        # We want the logit at position seq_len - 1
        pred_logits = logits[seq_len - 1]
        return optax.softmax_cross_entropy_with_integer_labels(pred_logits, y_i)
        
    losses = jax.vmap(single_example_loss)(x, y, mask)
    return jnp.mean(losses)

@eqx.filter_jit
def tf_step(model, opt_state, x, y, mask):
    loss, grads = eqx.filter_value_and_grad(tf_loss_fn)(model, x, y, mask)
    updates, opt_state = tf_optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

print("Training Baseline Transformer...")
tf_start_time = time.time()
tf_key = k2

tf_epochs = 10000
for epoch in range(tf_epochs):
    tf_key, subkey = jax.random.split(tf_key)
    x, y, mask = get_batch('train', batch_size, key=subkey)
    
    tf_model, tf_opt_state, loss = tf_step(tf_model, tf_opt_state, x, y, mask)
    
    if epoch % 500 == 0 or epoch == tf_epochs - 1:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Time: {time.time() - tf_start_time:.1f}s")


Training Baseline Transformer...
Epoch    0 | Loss: 4.7553 | Time: 6.0s
Epoch  500 | Loss: 2.6610 | Time: 17.5s
Epoch 1000 | Loss: 2.6093 | Time: 28.7s
Epoch 1500 | Loss: 2.5058 | Time: 40.4s
Epoch 2000 | Loss: 2.5640 | Time: 51.9s
Epoch 2500 | Loss: 2.1553 | Time: 63.7s
Epoch 3000 | Loss: 1.8139 | Time: 75.7s
Epoch 3500 | Loss: 2.1954 | Time: 87.3s
Epoch 4000 | Loss: 1.8710 | Time: 98.7s
Epoch 4500 | Loss: 1.9402 | Time: 110.6s
Epoch 5000 | Loss: 2.1698 | Time: 121.9s
Epoch 5500 | Loss: 2.0458 | Time: 133.5s
Epoch 6000 | Loss: 2.2614 | Time: 145.3s
Epoch 6500 | Loss: 2.1512 | Time: 156.8s
Epoch 7000 | Loss: 1.9839 | Time: 168.4s
Epoch 7500 | Loss: 2.0364 | Time: 180.4s
Epoch 8000 | Loss: 1.7998 | Time: 191.9s
Epoch 8500 | Loss: 1.4870 | Time: 203.2s
Epoch 9000 | Loss: 2.0097 | Time: 215.1s
Epoch 9500 | Loss: 1.9868 | Time: 226.9s
Epoch 9999 | Loss: 2.4313 | Time: 238.5s


In [27]:
def tf_generate(model, prompt, max_new_tokens=50):
    tokens = encode(prompt)
    
    for _ in range(max_new_tokens):
        # Convert to JAX array (seq_len,)
        x = jnp.array(tokens, dtype=jnp.int32)
        
        # The transformer expects a 1D sequence for a single example
        logits = model(x) # Shape: (seq_len, vocab_size)
        
        # Get the logits for the very last token in the sequence
        next_token_logits = logits[-1]
        
        # Greedy decoding (argmax)
        next_token = jnp.argmax(next_token_logits).item()
        
        tokens.append(next_token)
        
    return decode(tokens)

print("Transformer Generation After Training:")
out = tf_generate(tf_model, "ROMEO ")
print(out)

Transformer Generation After Training:
ROMEO SARIBE OF OF OF OR OF OF OF OR OF SERSE SERONE OF 
